<a href="https://colab.research.google.com/github/adrianacalderon/CaribbeanTraining/blob/main/2_Historical_Analogues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Historical Analogues**

This tutorial presents an example of seasonal streamflow forecasting using the ***historical analogues*** method. In summary, this method assumes that next month's hydrology is likely to reflect the conditions of the current month.

The scripts below are based on the methodology presented in:


*   *Svensson, C. (2016). Seasonal river flow forecasts for the United Kingdom using persistence and historical analogues. Hydrological Sciences Journal, 61(1), 19-35.*

The calculation example developed in this tutorial refers to the "*Weighted mean analogue forecast*" method. In this case, the forecast results from a weighted average of the analogues, where the weight is related to the error (RMSE) of the anomalies corresponding to these analogues with respect to the anomalies observed in the recent months.

---

# **Cloning the HydroSOS repository**


We will clone the WMO HydroSOS repository hosted on GitHub to collect a sample time series of daily streamflow.

Note: To use the git clone command on any Python notebook/script execution platform on your local machine (e.g., ***Jupyter***, ***Visual Studio Code***), you must make sure that Git is installed on your computer. Otherwise, Git can be downloaded from (https://git-scm.com/downloads)

In [ ]:
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Importing Python libraries**

Let's import the libraries needed for this tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime
from matplotlib import pyplot as plt

---

# **Create new folder**

We will automatically create an output folder in the root directory where we will export the results:

In [ ]:
# Define folder path (e.g., in the current working directory)
folder_output = 'output'

# Create the folder if it doesn't exist
os.makedirs(folder_output, exist_ok=True)

---

# **1. Define parameters**

Define parameters for generating the forecast based on historical analogues:

*   `File  path`: Path where the .csv file with daily streamflow is located
*   `fcst_range`: Range (in months) of the intended forecast
*   `N_analogs`: Maximum number of allowed analogues
*   `analog_length`: Duration (in months) of the historical sequence used to compare anomalies
*   `apply_transformation`: Indicates whether we want to apply a logarithmic transformation to the monthly streamflow (True/False)
*   `min_valid_frac`: Minimum fraction of valid daily data in a month required to convert them to a monthly average

*   `clim_startYear`: Start year of the reference period for calculating anomalies
*   `clim_endYear`: End year of the reference period for calculating anomalies


In [ ]:
file_path = './HydroSOS/example_data/forecast/input/obs_dir/obs_1.csv' # Path to your CSV file
fcst_range = 3                    # Forecast range (months ahead)
N_analogs = 5                     # Number of analogues from the past (ensemble members)
analog_length = 6                 # Length (in months) of the historical sequences to compare anomalies

apply_transformation = True       # Set to True to apply log transformation
min_valid_frac = 0.5              # Minimum fraction of valid daily data per month

clim_startYear = 1991             # Start year to compute anomalies (mean, standard deviation)
clim_endYear = 2020               # End year to compute anomalies

---

# **2. Read the data**

2.1. Let's read the .csv file containining the daily streamflow data and convert it to monthly data. The .csv file must have 2 columns (one with the dates, and the other one with streamflow values), as well as:

1.   The title of the date column defined as '*Date*' (without quotes) and the streamflow defined as '*Discharge*'
2.   The dates in **YYYY-MM-DD** format

In [ ]:
# Read and preprocess data
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')

df.set_index('Date', inplace=True)
df.columns = ['Discharge'] # ensure the name of the discharge column in csv

# Convert to monthly data
df_month = df.resample('MS').mean()

2.2. Plot streamflow:

In [ ]:
#@title Plot
# Plot on screen (subplot for daily and monthly data)
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes = axes.flatten()

df.plot(ax=axes[0], color='k', linewidth=0.5, title='Daily')
df_month.plot(ax=axes[1], color='k', linewidth=0.5, title='Monthly')

for ax in axes:
  ax.set_ylabel('Streamflow (m³/s)')
  ax.legend(['Observed'])

plt.tight_layout()
plt.show()

---

# **3. Pre-processing of monthly data**

3.1. Remove the months with insufficient daily data according to the minimum defined in the parameters section:



In [ ]:
# Resample daily to monthly step and compute de fraction of dates in each month without missing data
valid_fraction = df.resample('MS').apply(lambda x: x.count() / x.size)

# Replace monthly values to NaN if % of valid < minimum
df_month.mask(valid_fraction < min_valid_frac, np.nan, inplace=True)

3.2. Applying logarithmic transformation $y = log(Q)$ to the monthly streamflow series (if this option has been enabled by the user in the parameter definitions). This process should make the distribution of streamflow more similar to a normal distribution for calculating standardized monthly anomalies:

In [ ]:
# Apply log transformation if selected
if apply_transformation:
    if (df_month <= 0).sum().values > 0:
        raise ValueError("Log transformation requires all values to be > 0.")
    df_month_transformed = np.log(df_month)
else:
    df_month_transformed = df_month.copy()

3.3. Let's check the streamflow histogram to see the distribution:

In [ ]:
# Plot the discharge distribution in a histogram
df_month_transformed.hist()

---

# **4. Calculate standard anomalies**

4.1. Calculation of the mean $ (\overline{x}_m) $ and standard deviation $ (\sigma_m) $ for each calendar month. The statistics are calculated for the reference climatological period:

In [ ]:
# Compute monthly climatology (mean and std for each calendar month)
dt_start = datetime(clim_startYear, 1, 1)
dt_end = datetime(clim_endYear, 12, 31)

# Select period to compute anomalies
monthly_transformed_clim = df_month_transformed[(df_month_transformed.index >= dt_start) &
                                               (df_month_transformed.index <= dt_end)]

# Compute mean and standard deviation
month_nums = monthly_transformed_clim.index.month
clim_mean = monthly_transformed_clim.groupby(month_nums).mean()
clim_std = monthly_transformed_clim.groupby(month_nums).std()

4.2. Plotting the mean and standard deviation values:

In [ ]:
# For visualization purposes only
conc_data = pd.concat([clim_mean, clim_std], axis=1)
conc_data.columns = ['Mean', 'Std deviation']
conc_data.index.name = 'Month'
display(conc_data)

4.3. Calculation of standardized anomalies for each value in the monthly series. The calculation uses the mean and standard deviation of the month corresponding to the observed value, according to the following equation:  

\begin{align}
  a_{t,m} = \frac{q_{t,m}-\overline{x_m}}{\sigma_m}
\end{align}

Where:

*   $ a_{t,m} $ is the anomaly for time interval $t$ and month $m$;
*   $ q_{t,m} $ is the (transformed) streamflow for time interval $t$ and month $m$;
*   $ \overline{x_m} $ is the mean for month $m$;
*   $ \sigma_m $ is the standard deviation for month $m$.

In [ ]:
# Create a copy of the data series and update it to anomalies
df_anomalies = df_month_transformed.copy()

# Compute anomalies for each month separately
for i in range(1, 13):
  idx = (df_anomalies.index.month == i)
  df_anomalies.loc[idx] = (df_month_transformed.loc[idx] - clim_mean.loc[i].values) / clim_std.loc[i].values


4.4. Identify the anomalies for the recent months considering a sequence of defined duration (`analog_length`):

In [ ]:
# Get anomalies for the recent months
recent_anomalies = df_anomalies[-analog_length:]

4.5. Plotting the graph of anomalies over the last few months:

In [ ]:
#@title Plot
# Plot recent anomalies
recent_anomalies.plot(color='b', marker='o')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.ylabel('Standardized Anomaly')
plt.title(f'Observed anomalies over the last {analog_length} months')
plt.show()

---

# **5. Identification of potential analogues**

5.1. At this stage we will identify the candidate (i.e., potential) analogues to represent our forecast. In general, the sequence of data in each past year, for the same dates as the anomalies observed over the recent months, refers to a potential analogue, provided it does not contain invalid data. First, we must search the available historical record for all the dates corresponding to the calendar month in which the analogues begin:

In [ ]:
# Get the starting month and year of the period corresponding to recent anomalies
analog_start_month = recent_anomalies.index[-analog_length].month
analog_start_year = recent_anomalies.index[-analog_length].year

# Find dates of all calendar months (of previous years) = analog start month and < analog start year
start_indices = df_anomalies.index[(df_anomalies.index.month == analog_start_month) &
                                  (df_anomalies.index.year < analog_start_year)]

# Print starting dates of potential historical analogues
print('Start dates for potential analogues:')
display(start_indices)

5.2. We can build a table with the dates for the entire sequence of interest (considering the duration $ D_{ana} $ = `analog_length`) and each potential analogue:


In [ ]:
# Get the sequences of potential historical analog dates
analog_indices = {
    idx.year: pd.date_range(start=idx, periods=analog_length, freq='MS')
    for idx in start_indices}

# Generate dataframe
df_analog_dates = pd.DataFrame(analog_indices, index=np.arange(1, analog_length+1))

print('Dates for Potential Analogues:')
display(df_analog_dates)

5.3. Map the values of our monthly anomaly series to the previously identified dates in each analogue sequence:

In [ ]:
# Mapping anomalies from time series to their corresponding dates in table
df_analog_anomalies = df_analog_dates.map(lambda t: df_anomalies.loc[t, 'Discharge'] if t in df_anomalies.index else np.nan)

print('Anomalies for each potential analogue:')
display(df_analog_anomalies)

5.4. The same procedure is applied for the forecast dates and anomalies, which in practice refer to the extension of the potential analogue sequences to later dates $ (D_{ana}+m) $, where $m$ is a month index ranging from 1 to `fcst_range`:

In [ ]:
# Select dates corresponding to the last row of potential analog sequences
start_fcst_indices = df_analog_dates.iloc[-1]

# For each column, generate forecast ranges starting from the value in the last row (w/ one month offset)
# This is the same as (Dana + m)
fcst_indices = {
    col: pd.date_range(start=start_fcst_indices[col] + pd.DateOffset(months=1), periods=fcst_range, freq='MS')  # MS = Month Start
    for col in df_analog_dates.columns}

# Create dataframe of historic forecast dates
df_fcst_analog_dates = pd.DataFrame(fcst_indices, index=np.arange(1, fcst_range+1))
# Map anomalies to their corresponding dates
df_fcst_analog_anom = df_fcst_analog_dates.map(lambda t: df_anomalies.loc[t, 'Discharge'] if t in df_anomalies.index else np.nan)

print('Forecast dates related to the potential analogues:')
display(df_fcst_analog_dates)
print('Forecast anomalies for each potential analogue:')
display(df_fcst_analog_anom)

5.5. When there is missing data in the sequence of the potential analogue or the corresponding forecast, the analogue must be excluded from the list:

In [ ]:
# Check if there are missing months in analogs or analog forecasts
analog_missing = np.isnan(df_analog_anomalies).sum(axis=0)
fcst_missing = np.isnan(df_fcst_analog_anom).sum(axis=0)

# If there are years with missing data either in analog or its corresponding forecast, drop them
missing_years = analog_missing + fcst_missing
if missing_years.sum() > 0:
  idx = missing_years[missing_years > 0].index
  df_analog_anomalies = df_analog_anomalies.drop(idx, axis=1)
  df_fcst_analog_anom = df_fcst_analog_anom.drop(idx, axis=1)

display(df_fcst_analog_anom)


---


# **6. Evaluation of potential analogues**

6.1. Next, we will evaluate the suitability of each analogue as future information. To do this, we calculate the RMSE between the recent anomalies and the anomalies of each potential analogue's historical sequence:

\begin{align}
  RMSE = \sqrt{\frac{1}{D_{ana}}\sum_{k=1}^{D_{ana}}{(a_p(k)-a_r(k))^2}}
\end{align}

Where:

*   $ D_{ana} $ is the duration of the analogue;
*   $ a_p(k) $ is the streamflow anomaly for each month *k* in the potential analogue;
*   $ a_r(k) $ is the corresponding anomaly in the recent past.


In [ ]:
rmse_values = []
# Year loop
for col in df_analog_anomalies.columns:

  # Get historical sequence
  historical_sequence = df_analog_anomalies[col].values
  current_sequence = recent_anomalies.values.flatten()

  # Compute RMSE
  rmse = np.sqrt(np.mean((current_sequence - historical_sequence)**2.0))

  # Include the computed RMSE in list
  rmse_values.append(rmse)

df_rmse = pd.DataFrame(rmse_values, index=df_analog_anomalies.columns, columns=['RMSE'])

6.2. Plotting the graph of the differences (RMSE) between the anomalies of the recent sequence and each potential analogue (historical year).


In [ ]:
#@title Plot
# Plot RMSE of analogs from past years
df_rmse['RMSE'].plot(figsize=(12,4))
plt.ylabel('RMSE (-)')
plt.title('RMSE between the recent anomalies and the anomalies of each potential analogue (start year)')
plt.show()

---

# **7. Final selection of historical analogues**

7.1. We will select the `N_analogs` with the smallest differences (RMSE) relative to the recent anomaly sequence:

In [ ]:
# Sort dataframe by rmse values (ascending order)
df_rmse_sorted = df_rmse.sort_values(by='RMSE', ascending=True)

# Select only the first N analogs (with lower RMSE)
df_rmse_sorted = df_rmse_sorted.iloc[0:N_analogs]

print('Selected analogues:')
display(df_rmse_sorted.head(N_analogs))

7.2. After selecting the historical analogues with the lowest RMSE, we calculate the weights according to the following equation:

\begin{align}
  w(b) = \frac{1}{RMSE(b)} / \sum_{b=1}^{N_{ana}} \frac{1}{RMSE(b)}
\end{align}

Where:

*   $ w(b) $ is the weight of analogue $b$;
*   $ N_{ana} $ is the number of selected analogues.

In [ ]:
# Compute weights for each analog based on its (inverse) RMSE
rmse_sum = (1.0 / df_rmse_sorted['RMSE']).sum()
df_rmse_sorted['weight'] = (1.0 / df_rmse_sorted['RMSE']) / rmse_sum

print('Selected analogues and weights:')
display(df_rmse_sorted.head(N_analogs))

print(f'Sum of the weights: {df_rmse_sorted["weight"].sum(axis=0)}')

---


# **8. Calculation of the analogue-based forecast**

8.1. We will look up the "forecast" anomaly series corresponding to the selected analogues (those with the lowest RMSE):

In [ ]:
# Retrieving only the analog forecast sequences corresponding to years with lowest RMSE
fcst_selected = df_fcst_analog_anom[df_rmse_sorted.index]

display(fcst_selected)

8.2. Calculation of the average forecast anomaly weighted by the RMSE, using the selected analogues:

\begin{align}
  a_f(m) = \sum_{b=1}^{N_{ana}}{w(b)a_{p,b}(D_{ana}+m)}
\end{align}

Where:

*   $ a_f(m) $ is the weighted forecast anomaly, for month $m$;
*   $ a_{p,b}(D_{ana}+m) $ is the forecast anomaly corresponding to analogue $b$, for month $m$;


In [ ]:
# Get weights
weights = df_rmse_sorted['weight']
# Multiply matrices (analogues x weights) and sum values
df_weighted_anomaly = fcst_selected.mul(weights).sum(axis=1)

print('Forecast anomalies (weighted by RMSE):')
display(df_weighted_anomaly)

8.3. Identify the most recent date:

In [ ]:
# Check standardized anomaly for last month
most_recent_date = recent_anomalies.index[-1]

print(f'Most recent month of observations: {most_recent_date.strftime("%Y-%m")}')

8.4. We identify the dates of the *N* forecast months and calculate the streamflow from the weighted forecast anomalies, using the equation:

\begin{align}
  q_m = \overline{x_m} + (a_m \times \sigma_m)
\end{align}

Where:

*   $ q_{m} $ is the (transformed) forecasted streamflow for month $m$;
*   $ \overline{x_m} $ is the mean for month $m$;
*   $ a_{m} $ is the forecast anomaly for month $m$;
*   $ \sigma_m $ is the standard deviation for month $m$.

In [ ]:
# Generate forecast dates
forecast_dates = pd.date_range(start=most_recent_date + pd.offsets.MonthEnd(1),
                               periods=fcst_range,
                               freq='MS')

# Get forecast months
forecast_month_nums = forecast_dates.month

# Get the climatological mean and standard deviation for these months
clim_mean_fcst = clim_mean.loc[forecast_month_nums].values.flatten()
clim_std_fcst = clim_std.loc[forecast_month_nums].values.flatten()

# Reconstruct forecast values from anomalies, based on climatological means and standard deviations
forecast_transformed = clim_mean_fcst + (df_weighted_anomaly.values * clim_std_fcst)

# Construct dataframe
df_forecast_transformed = pd.DataFrame(forecast_transformed, index=forecast_dates, columns=['Discharge'])
df_forecast_transformed.index.name = 'Date'

print('Streamflow forecast (transformed)::')
display(df_forecast_transformed)

8.5. For plotting purposes, we can also obtain the streamflows of the analogues for the entire date range of interest, that is, for $D_{ana}$ + `fcst_range`:

In [ ]:
# Concatenating dates: selected analogs and corresponding "forecasts"
df_analog_dates_full = pd.concat([df_analog_dates, df_fcst_analog_dates], axis=0)

# Mapping (transfromed) discharges from time series to their corresponding dates in table
df_analog_full = df_analog_dates_full.map(lambda t: df_month_transformed.loc[t, 'Discharge'] if t in df_month_transformed.index else np.nan)

# Get only the N_analogs with lowest RMSE
df_analog_full = df_analog_full[df_rmse_sorted.index]

analog_dates = pd.date_range(start=recent_anomalies.index[0],
                               periods= (analog_length + fcst_range),
                               freq='MS')

df_analog_full.index = analog_dates
df_analog_full.index.name = 'Date'

print('Streamflow forecast (transformed)::')
display(df_analog_full)

8.6. Converting the streamflows back to their original format via $ ( Q = e^y ) $ (in case they were previously transformed):

In [ ]:
# Back-transform if log transformation was applied
if apply_transformation:
    df_forecast = np.exp(df_forecast_transformed)
    df_analogs = np.exp(df_analog_full)
else:
    df_forecast = df_forecast_transformed.copy()
    df_analogs = df_analog_full.copy()

df_forecast[df_forecast < 0.0] = 0.0

8.7. Plotting the final graph with the forecasts:

In [ ]:
#@title Plot
# Plot on screen
fig, ax = plt.subplots(figsize=(15, 4))

df_month.iloc[-36:].plot(ax=ax, color='k')
df_analogs.plot(ax=ax, color='gray', alpha=0.5)
df_forecast.plot(ax=ax, color='r', linewidth=2.5, linestyle='--')

# This part is only for visualization purposes
# Adjust legend labels
h, lbl = ax.get_legend_handles_labels()
lbl[0] = 'Observed'
lbl[-1] = 'Forecast'
ax.legend(h, lbl)

ax.set_ylabel('Streamflow (m³/s)')
ax.set_title('Historical Analogues')

plt.show()

---

# **9. Exporting the generated forecasts**

Finally, we can export our streamflow forecasts to the specified `output` folder:

In [ ]:
# Save to CSV
file_name = Path(file_path).stem
df_forecast.to_csv(f'{folder_output}/analog_forecast_{file_name}.csv')